# Limpieza reproducible · cuaderno del proyecto (sesión 5)
**Estud-IA · Análisis de Datos Básico · Equipo E8**

Este cuaderno **parte siempre del original** (`datos/originales/Gestion_de_Incidentes_agi2-ekag.csv`) y aplica las reglas de limpieza una por una. Cada regla anota sola cuántas filas afecta. Al final, los conteos tienen que cuadrar: **original = limpias + quitadas + a revisión**.

Corre las celdas en orden con ▶ (o Shift + Enter). Si Colab avisa que el cuaderno viene de GitHub, elige **«Ejecutar de todos modos»**.

**Tres reglas de oro**
1. El archivo de `datos/originales/` **no se toca**. Todo sale de aquí, con código.
2. Una fila que no sabes cómo arreglar **no se inventa ni se borra en silencio**: va **a revisión**.
3. Si te equivocas o cambias una regla: menú **Entorno de ejecución › Reiniciar y ejecutar todo**. Tiene que dar lo mismo cada vez.


In [1]:
# Celda 1 · Preparar (no cambies nada)
try:
    import duckdb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb
import re
import pandas as pd

def cargar(ruta, hoja=0):
    """Lee un CSV (coma o punto y coma) o un Excel. Acepta el enlace Raw o el de la página de GitHub."""
    ruta = re.sub(r"^https://github\.com/([^/]+)/([^/]+)/blob/", r"https://raw.githubusercontent.com/\1/\2/", ruta.strip())
    if ruta.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(ruta, sheet_name=hoja)
    for codificacion in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(ruta, sep=None, engine="python", encoding=codificacion)
        except UnicodeDecodeError:
            continue
        except pd.errors.ParserError:
            raise ValueError("No pude leerlo como tabla. Revisa que el enlace sea el de tu CSV (botón Raw).") from None
    raise ValueError("No pude leer el archivo: revisa que sea un CSV")

def sql(consulta):
    """Corre una consulta SQL sobre las tablas cargadas (original, base, analisis) y muestra el resultado."""
    resultado = duckdb.sql(consulta).df()
    for col in resultado.select_dtypes("float").columns:
        if resultado[col].dropna().mod(1).eq(0).all():
            resultado[col] = resultado[col].astype("Int64")   # 45000 en lugar de 45000.0
    return resultado

def _filas(n):
    return f"{n} fila" if n == 1 else f"{n} filas"

def perfil(tabla):
    """Cuántas filas y columnas hay, cuántas filas repetidas y cuántos vacíos tiene cada columna."""
    print(f"{_filas(len(tabla))} · {len(tabla.columns)} columnas · filas idénticas a otra: {int(tabla.duplicated().sum())}")
    return pd.DataFrame({
        "tipo": [str(t) for t in tabla.dtypes],
        "vacíos": tabla.isna().sum(),
        "valores distintos": tabla.nunique(),
        "ejemplo": [tabla[c].dropna().iloc[0] if tabla[c].notna().any() else "" for c in tabla.columns],
    })

def empezar(tabla):
    """Copia el original y deja la bitácora en cero. Todo vuelve a empezar desde aquí."""
    global base, quitadas, revision, bitacora, _original
    _original = tabla.copy()
    base = tabla.copy()
    quitadas = tabla.iloc[0:0].copy()
    revision = tabla.iloc[0:0].copy()
    bitacora = []
    print(f"Empezamos desde el original: {len(base)} filas.")

def _ya_esta(nombre):
    if any(b["regla"] == nombre for b in bitacora):
        raise RuntimeError(f"La regla «{nombre}» ya se aplicó. Corre otra vez desde la celda 4 (Empezar) para no aplicarla dos veces.")

def regla(nombre, filas, motivo, accion="quitar"):
    """Aplica una regla a las filas marcadas con True.
    accion = "quitar": salen de la base (por ejemplo, copias exactas).
    accion = "revisar": salen de la base y van al archivo de revisión (no se inventan ni se pierden)."""
    global base, quitadas, revision
    _ya_esta(nombre)
    filas = pd.Series(filas, index=base.index).fillna(False).astype(bool)
    n, antes = int(filas.sum()), len(base)
    afectadas = _original.loc[base.index[filas]].copy()   # con los valores originales, para poder revisarlas
    afectadas["regla"] = nombre
    if accion == "quitar":
        quitadas = pd.concat([quitadas, afectadas])
    elif accion == "revisar":
        revision = pd.concat([revision, afectadas])
    else:
        raise ValueError('accion debe ser "quitar" o "revisar"')
    base = base[~filas].copy()
    bitacora.append({"regla": nombre, "filas afectadas": n, "acción": accion, "motivo": motivo, "antes": antes, "después": len(base)})
    print(f"{nombre}: {_filas(n)} ({accion}). Antes {antes}, ahora {len(base)}.")
    return afectadas.head(10)

def unificar(columna, motivo="El mismo valor estaba escrito de varias formas"):
    """Quita espacios sobrantes y une las formas que solo cambian en mayúsculas: deja la forma más común."""
    nombre = f"Unificar escritura de {columna}"
    _ya_esta(nombre)
    antes = base[columna].copy()
    texto = antes.astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
    clave = texto.str.lower()
    mas_comun = texto.groupby(clave).agg(lambda s: s.value_counts().index[0])
    nuevo = clave.map(mas_comun)
    nuevo = nuevo.where(antes.notna(), antes)
    cambiadas = (antes.astype("string") != nuevo.astype("string")).fillna(False)
    base[columna] = nuevo
    bitacora.append({"regla": nombre, "filas afectadas": int(cambiadas.sum()), "acción": "corregir", "motivo": motivo, "antes": len(base), "después": len(base)})
    print(f"{nombre}: filas corregidas: {int(cambiadas.sum())}. Grupos antes: {antes.nunique()}, después: {base[columna].nunique()}.")
    return pd.DataFrame({"antes": antes[cambiadas], "después": nuevo[cambiadas]}).drop_duplicates().head(10)

def reemplazar(columna, cambios, motivo="El mismo valor estaba escrito de otra forma"):
    """Cambia valores exactos por otros: reemplazar("municipio", {"Estrella": "La Estrella"}).
    Sirve cuando unificar() no alcanza: tildes, abreviaturas o nombres distintos de la misma cosa."""
    nombre = f"Reemplazar en {columna}: " + ", ".join(f"{a} -> {b}" for a, b in cambios.items())
    _ya_esta(nombre)
    antes = base[columna].copy()
    base[columna] = base[columna].replace(cambios)
    cambiadas = (antes.astype("string") != base[columna].astype("string")).fillna(False)
    bitacora.append({"regla": nombre, "filas afectadas": int(cambiadas.sum()), "acción": "corregir", "motivo": motivo, "antes": len(base), "después": len(base)})
    print(f"{nombre}: filas corregidas: {int(cambiadas.sum())}. Valores distintos antes: {antes.nunique()}, después: {base[columna].nunique()}.")
    return pd.DataFrame({"antes": antes[cambiadas], "después": base[columna][cambiadas]}).value_counts().rename("filas").reset_index()

def a_numero(columna, miles=None):
    """Convierte una columna a número. Si los miles vienen con punto o coma, escribe miles="." o miles=",".
    Devuelve True en las filas que NO se pudieron convertir (y no estaban vacías): decide qué hacer con ellas."""
    texto = base[columna].astype("string").str.strip()
    if miles:
        texto = texto.str.replace(miles, "", regex=False)
    numero = pd.to_numeric(texto, errors="coerce")
    malas = numero.isna() & base[columna].notna()
    if numero.dropna().mod(1).eq(0).all():
        numero = numero.astype("Int64")   # enteros sin decimales
    base[columna] = numero
    print(f"{columna} ahora es número. Filas que no se pudieron convertir: {int(malas.sum())}.")
    return malas

def a_fecha(columna, formato):
    """Convierte una columna a fecha con un formato explícito (por ejemplo "%Y-%m-%d").
    Devuelve True en las filas que NO se pudieron convertir (y no estaban vacías)."""
    fecha = pd.to_datetime(base[columna].astype("string").str.strip(), format=formato, errors="coerce")
    malas = fecha.isna() & base[columna].notna()
    base[columna] = fecha
    print(f"{columna} ahora es fecha. Filas que no se pudieron leer con el formato {formato}: {int(malas.sum())}.")
    return malas

def ver_bitacora():
    """La bitácora de limpieza como tabla y como líneas para pegar en bitacora.md."""
    tabla = pd.DataFrame(bitacora)
    for b in bitacora:
        print(f"- {b['regla']} · {_filas(b['filas afectadas'])} · {b['acción']} · {b['motivo']}")
    return tabla

def conciliar(original):
    """Comprueba que no se perdió ninguna fila: original = limpias + quitadas + a revisión."""
    total = len(base) + len(quitadas) + len(revision)
    print(f"Original {len(original)} = limpias {len(base)} + quitadas {len(quitadas)} + a revisión {len(revision)}  →  {total}")
    print("Cuadra." if total == len(original) else "NO CUADRA: alguna fila se perdió o se duplicó. Corre todo desde la celda 4.")

def guardar(nombre):
    """Guarda la base limpia y el archivo de revisión como CSV y los descarga a tu computador."""
    base.to_csv(f"{nombre}_limpio.csv", index=False)
    revision.to_csv(f"{nombre}_revision.csv", index=False)
    print(f"Guardé {nombre}_limpio.csv ({len(base)} filas) y {nombre}_revision.csv ({len(revision)} filas).")
    try:
        from google.colab import files
        files.download(f"{nombre}_limpio.csv")
        files.download(f"{nombre}_revision.csv")
    except Exception:
        print("Fuera de Colab: los archivos quedaron en la carpeta del cuaderno.")

print("Listo. Sigue con la celda 2.")

Listo. Sigue con la celda 2.


## 1. Carga el ORIGINAL

Enlace del archivo congelado en **`datos/originales/`** (4.243 filas y 9 columnas).


In [2]:
# Celda 2 · Cargar el original (no se modifica)
ENLACE = "https://raw.githubusercontent.com/wiguelJM/proyecto-E8/main/datos/originales/Gestion_de_Incidentes_agi2-ekag.csv"
HOJA = 0   # solo para Excel con varias hojas: el nombre entre comillas

original = cargar(ENLACE, HOJA)
print(f"Original: {len(original)} filas y {len(original.columns)} columnas.")
original.head()

Original: 4243 filas y 9 columnas.


,FECHA DE SOLICITUD,ORDEN DE LA ENTIDAD,SECTOR,REGIÓN,DIVIPOLA DEPARTAMENTO,DEPARTAMENTO,CLASIFICACIÓN TAXONOMÍA,TIPO DE INCIDENTE,IMPORTANCIA
0,03/22/2019 12:00:00 AM,Nacional,Deporte,Bogotá D.C.,11,Bogotá D.C.,Intrusión,Compromiso de Aplicaciones,Grave
1,04/07/2019 12:00:00 AM,Territorial,Planeación,Eje Cafetero,5,Antioquia,Intrusión,Compromiso de Aplicaciones,Grave
2,04/15/2019 12:00:00 AM,Territorial,Planeación,Llanos Orientales,50,Meta,Contenido Dañino,Sistema Infectado,Menor
3,04/16/2019 12:00:00 AM,Territorial,Planeación,Llanos Orientales,85,Casanare,Compromiso de Información,Modificación no Autorizada de Información,Menor
4,05/03/2019 12:00:00 AM,Territorial,Planeación,Eje Cafetero,5,Antioquia,Fraude,Suplantación,Menor


## 2. Mira antes de limpiar

El perfil te dice qué revisar: filas idénticas, columnas con vacíos y columnas que deberían ser número o fecha pero salen como texto (`object` o `str`).


In [3]:
# Celda 3 · Perfil del original
perfil(original)

4243 filas · 9 columnas · filas idénticas a otra: 856


,tipo,vacíos,valores distintos,ejemplo
FECHA DE SOLICITUD,str,0,1160,03/22/2019 12:00:00 AM
ORDEN DE LA ENTIDAD,str,0,3,Nacional
SECTOR,str,0,27,Deporte
REGIÓN,str,0,8,Bogotá D.C.
DIVIPOLA DEPARTAMENTO,int64,0,33,11
DEPARTAMENTO,str,0,33,Bogotá D.C.
CLASIFICACIÓN TAXONOMÍA,str,0,12,Intrusión
TIPO DE INCIDENTE,str,0,27,Compromiso de Aplicaciones
IMPORTANCIA,str,0,4,Grave


## 3. Empieza (siempre desde el original)

Cada vez que cambies una regla, vuelve a correr **desde aquí** hacia abajo. Si corres una regla dos veces, el cuaderno te avisa.

In [4]:
# Celda 4 · Empezar desde el original
empezar(original)

Empezamos desde el original: 4243 filas.


## 4. Tus reglas, una por celda

Cada regla tiene **nombre**, las **filas** que afecta (True/False), un **motivo** y una **acción**:
- `"quitar"`: salen de la base. Solo para lo que estás seguro de que sobra (una copia exacta con identificador único).
- `"revisar"`: salen de la base y van al archivo de revisión. Para lo que no sabes cómo arreglar sin inventar (una fecha imposible).
- `unificar(...)`: corrige mayúsculas/minúsculas y espacios sin perder ninguna fila.

**Decisión sobre las 856 filas idénticas (límite documentado):** `perfil(original)` detecta 856 filas idénticas a otra, pero esta base **no tiene código único de incidente ni hora real** (todas las fechas traen `12:00:00 AM`). Dos filas iguales pueden ser incidentes distintos ocurridos el mismo día en el mismo sector y tipo. Por eso **NO se quitan** y se documenta como límite metodológico.


In [5]:
# Celda 5 · Regla 1: unificar mayúsculas en SECTOR
unificar("SECTOR",
         motivo="Unificar mayúsculas en 'Ambiente Y/y Desarrollo Sostenible' para evitar sectores duplicados")

Unificar escritura de SECTOR: filas corregidas: 3. Grupos antes: 27, después: 26.


,antes,después
871,Ambiente Y Desarrollo Sostenible,Ambiente y Desarrollo Sostenible


In [6]:
# Celda 6 · Regla 2: unificar mayúsculas en CLASIFICACIÓN TAXONOMÍA
unificar("CLASIFICACIÓN TAXONOMÍA",
         motivo="Unificar mayúsculas en 'Contenido Dañino/dañino' e 'Intento de Intrusión/intrusión' para evitar categorías duplicadas")

Unificar escritura de CLASIFICACIÓN TAXONOMÍA: filas corregidas: 2. Grupos antes: 12, después: 10.


,antes,después
3799,Contenido dañino,Contenido Dañino
4071,Intento de intrusión,Intento de Intrusión


In [7]:
# Celda 7 · Regla 3: unificar mayúsculas en TIPO DE INCIDENTE
unificar("TIPO DE INCIDENTE",
         motivo="Unificar mayúsculas en 'Mala Configuración/configuración' para evitar tipos duplicados")

Unificar escritura de TIPO DE INCIDENTE: filas corregidas: 1. Grupos antes: 27, después: 26.


,antes,después
1432,Mala Configuración,Mala configuración


### Fechas: primero se convierte, después se decide

En `FECHA DE SOLICITUD` las fechas vienen como `03/22/2019 12:00:00 AM` (`"%m/%d/%Y %I:%M:%S %p"`). Las que no se puedan leer van a revisión con una regla.


In [8]:
# Celda 8 · Regla 4: convertir FECHA DE SOLICITUD y mandar imposibles a revisión
malas = a_fecha("FECHA DE SOLICITUD", "%m/%d/%Y %I:%M:%S %p")
regla("Fecha imposible",
      malas,
      motivo="Fechas que no se pudieron leer con el formato MM/DD/AAAA HH:MM:SS AM/PM; no se inventan",
      accion="revisar")

FECHA DE SOLICITUD ahora es fecha. Filas que no se pudieron leer con el formato %m/%d/%Y %I:%M:%S %p: 0.
Fecha imposible: 0 filas (revisar). Antes 4243, ahora 4243.


,FECHA DE SOLICITUD,ORDEN DE LA ENTIDAD,SECTOR,REGIÓN,DIVIPOLA DEPARTAMENTO,DEPARTAMENTO,CLASIFICACIÓN TAXONOMÍA,TIPO DE INCIDENTE,IMPORTANCIA,regla


## 5. ¿Cuadra?

**Original = limpias + quitadas + a revisión.** Si no cuadra, alguna regla perdió o duplicó filas: vuelve a correr desde la celda 4.

Para nuestra base: **4243 = 4243 + 0 + 0** (cuadra).


In [9]:
# Celda 9 · Conciliar los conteos
conciliar(original)

Original 4243 = limpias 4243 + quitadas 0 + a revisión 0  →  4243
Cuadra.


In [10]:
# Celda 10 · La bitácora de limpieza (copia las líneas a bitacora.md, sección Limpieza)
ver_bitacora()

- Unificar escritura de SECTOR · 3 filas · corregir · Unificar mayúsculas en 'Ambiente Y/y Desarrollo Sostenible' para evitar sectores duplicados
- Unificar escritura de CLASIFICACIÓN TAXONOMÍA · 2 filas · corregir · Unificar mayúsculas en 'Contenido Dañino/dañino' e 'Intento de Intrusión/intrusión' para evitar categorías duplicadas
- Unificar escritura de TIPO DE INCIDENTE · 1 fila · corregir · Unificar mayúsculas en 'Mala Configuración/configuración' para evitar tipos duplicados
- Fecha imposible · 0 filas · revisar · Fechas que no se pudieron leer con el formato MM/DD/AAAA HH:MM:SS AM/PM; no se inventan


,regla,filas afectadas,acción,motivo,antes,después
0,Unificar escritura de SECTOR,3,corregir,Unificar mayúsculas en 'Ambiente Y/y Desarroll...,4243,4243
1,Unificar escritura de CLASIFICACIÓN TAXONOMÍA,2,corregir,Unificar mayúsculas en 'Contenido Dañino/dañin...,4243,4243
2,Unificar escritura de TIPO DE INCIDENTE,1,corregir,Unificar mayúsculas en 'Mala Configuración/con...,4243,4243
3,Fecha imposible,0,revisar,Fechas que no se pudieron leer con el formato ...,4243,4243


In [11]:
# Celda 11 · Lo que quedó a revisión (no se pierde: queda en su propio archivo)
revision

,FECHA DE SOLICITUD,ORDEN DE LA ENTIDAD,SECTOR,REGIÓN,DIVIPOLA DEPARTAMENTO,DEPARTAMENTO,CLASIFICACIÓN TAXONOMÍA,TIPO DE INCIDENTE,IMPORTANCIA,regla


## 6. Base limpia frente a vista de análisis

La **base limpia** tiene todas las 4.243 filas válidas (incluidas las 121 filas con `IMPORTANCIA = 'Solicitud'`). La **vista de análisis** excluye `'Solicitud'` (4.122 filas) porque no representa un nivel de gravedad, evitando distorsionar el denominador de `% graves y muy graves`. No se borra lo demás: se filtra.


In [12]:
# Celda 12 · Vista de análisis: excluir 'Solicitud' para indicadores de gravedad
analisis = base[base["IMPORTANCIA"] != "Solicitud"].copy()
print(f"Base limpia: {len(base)} filas · vista de análisis (sin Solicitud): {len(analisis)} filas")

Base limpia: 4243 filas · vista de análisis (sin Solicitud): 4122 filas


## 7. Vuelve a correr tus consultas sobre la base limpia

Las consultas de la sesión 3 ahora van sobre `analisis` (o `base`) en lugar de `datos`. ¿Qué cifra cambió frente a la que tenías? Anótalo en la bitácora: **antes · después · por qué**.

In [13]:
# Celda 13 · Comparativa antes (original) vs después (base limpia / vista de análisis)
antes_global = sql('''
SELECT COUNT(*) AS filas,
       SUM(CASE WHEN "IMPORTANCIA" IN ('Grave', 'Muy Grave') THEN 1 ELSE 0 END) AS graves_y_muy_graves,
       ROUND(100.0 * SUM(CASE WHEN "IMPORTANCIA" IN ('Grave', 'Muy Grave') THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_graves
FROM original
''')
despues_global = sql('''
SELECT COUNT(*) AS filas,
       SUM(CASE WHEN "IMPORTANCIA" IN ('Grave', 'Muy Grave') THEN 1 ELSE 0 END) AS graves_y_muy_graves,
       ROUND(100.0 * SUM(CASE WHEN "IMPORTANCIA" IN ('Grave', 'Muy Grave') THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_graves
FROM analisis
''')
print("Global antes (original, incluye Solicitud):", antes_global.to_dict(orient="records")[0])
print("Global después (analisis, sin Solicitud):", despues_global.to_dict(orient="records")[0])

amb_antes = int((original["SECTOR"] == "Ambiente y Desarrollo Sostenible").sum())
amb_antes_y = int((original["SECTOR"] == "Ambiente Y Desarrollo Sostenible").sum())
amb_despues = int((base["SECTOR"] == "Ambiente y Desarrollo Sostenible").sum())
print(f"Ambiente y Desarrollo Sostenible: antes {amb_antes} + {amb_antes_y} separados → después {amb_despues} unificado")

sql('''
SELECT "SECTOR",
       COUNT(*) AS incidentes,
       SUM(CASE WHEN "IMPORTANCIA" IN ('Grave', 'Muy Grave') THEN 1 ELSE 0 END) AS graves_y_muy_graves,
       ROUND(100.0 * SUM(CASE WHEN "IMPORTANCIA" IN ('Grave', 'Muy Grave') THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_graves
FROM analisis
GROUP BY "SECTOR"
ORDER BY incidentes DESC
''')

Global antes (original, incluye Solicitud): {'filas': 4243, 'graves_y_muy_graves': 1739, 'pct_graves': 40.99}
Global después (analisis, sin Solicitud): {'filas': 4122, 'graves_y_muy_graves': 1739, 'pct_graves': 42.19}
Ambiente y Desarrollo Sostenible: antes 67 + 3 separados → después 70 unificado


,SECTOR,incidentes,graves_y_muy_graves,pct_graves
0,Planeación,1252,446,35.62
1,Privado,1104,428,38.77
2,Salud,210,97,46.19
3,Hacienda y Crédito Público,181,50,27.62
4,Tecnologías de la Información y Comunicaciones,167,69,41.32
5,Educación,149,85,57.05
6,Justicia y Derecho,129,42,32.56
7,Defensa,117,58,49.57
8,Transporte,114,62,54.39
9,"Vivienda, Ciudad y Territorio",88,63,71.59


## 8. Guarda y sube por PR

1. Corre la celda 14: descarga `…_limpio.csv` y `…_revision.csv`.
2. Menú **Archivo › Descargar › Descargar .ipynb** (este cuaderno, con tus reglas).
3. En GitHub, en una **rama nueva**: los dos CSV a `datos/limpios/` y el cuaderno a `notebooks/02_limpieza.ipynb` (**Add file › Upload files**).
4. Las líneas de la celda 10 a `bitacora.md` (sección **Limpieza**) y, si usaste IA, a **Uso de IA**.
5. Abre el PR: en la descripción, la conciliación (**original = limpias + quitadas + revisión**) y pide revisión a **santiagom26cc**.

**Prueba final:** menú **Entorno de ejecución › Reiniciar y ejecutar todo**. ¿Te da los mismos números? Entonces es reproducible.

In [14]:
# Celda 14 · Guardar CSV limpio y de revisión
guardar("incidentes")

Guardé incidentes_limpio.csv (4243 filas) y incidentes_revision.csv (0 filas).
Fuera de Colab: los archivos quedaron en la carpeta del cuaderno.


---
¿Algo falló? Regla de 3 y, si sigues bloqueado, un ticket en la **Cola de ayuda** con el mensaje de error copiado. Nunca pegues contraseñas ni datos personales.